In [6]:

import json
import time
import requests
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------

BASE_URL = "https://gateway.reemioltd.com/api/v2"

# Request a large limit; the server may enforce a lower maximum.
REQUESTED_LIMIT = 5000
MAX_PAGES = 1000
TIMEOUT = (15, 120)
CHECKPOINT_EVERY = 10

OUTPUT_DIR = Path("reemio_data")
OUTPUT_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------
# HTTP SESSION
# ---------------------------------------------------------

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0",
    "Origin": "https://listings.reemioltd.com",
    "Referer": "https://listings.reemioltd.com/",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
})

retry = Retry(
    total=5,
    connect=5,
    read=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    respect_retry_after_header=True,
)

adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------

def get_json(url, params=None):
    """Fetch JSON without browser-supplied conditional cache headers."""
    response = session.get(
        url,
        params=params,
        timeout=TIMEOUT,
    )

    if response.status_code == 304:
        raise RuntimeError(
            "Received HTTP 304 without a JSON body. "
            "Check whether a conditional cache header is being added."
        )

    response.raise_for_status()
    return response.json()


def extract_records(payload):
    """
    Support common API response formats:
    {content: [...]}, {data: [...]}, {results: [...]}, or [...]
    """
    if isinstance(payload, list):
        return payload

    if not isinstance(payload, dict):
        return None

    for key in ("content", "data", "results", "listings", "areas"):
        value = payload.get(key)

        if isinstance(value, list):
            return value

        if isinstance(value, dict):
            nested = extract_records(value)
            if nested is not None:
                return nested

    return None


def save_records(name, records):
    """Save original nested JSON and a flattened CSV."""
    json_path = OUTPUT_DIR / f"{name}.json"
    csv_path = OUTPUT_DIR / f"{name}.csv"

    with open(json_path, "w", encoding="utf-8") as file:
        json.dump(
            records,
            file,
            ensure_ascii=False,
            indent=2,
        )

    if records:
        df = pd.json_normalize(records, sep="_")
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    else:
        df = pd.DataFrame()

    print(f"{name}: {len(records):,} records")
    print(f"  JSON: {json_path}")
    if records:
        print(f"  CSV:  {csv_path}")

    return df


def fetch_paginated(
    name,
    endpoint,
    base_params=None,
    limit=REQUESTED_LIMIT,
    max_pages=MAX_PAGES,
):
    """
    Collect records from a paginated endpoint.

    Uses totalPages when returned by the API, with a hard
    page cap and an empty-page stopping condition.
    """
    base_params = dict(base_params or {})
    base_params.setdefault("limit", limit)

    first_payload = None
    records = []
    seen_ids = set()
    errors = []

    print(f"\n{'=' * 70}")
    print(f"FETCHING: {name.upper()}")
    print(f"Endpoint: {endpoint}")
    print(f"Requested limit: {base_params.get('limit')}")
    print("=" * 70)

    # Discover pagination metadata from page 1.
    try:
        first_payload = get_json(
            f"{BASE_URL}/{endpoint.lstrip('/')}",
            params={**base_params, "page": 1},
        )
    except Exception as exc:
        print(f"Initial request failed: {exc}")
        return records, errors

    if isinstance(first_payload, dict):
        reported_total = first_payload.get("totalElements")
        reported_pages = first_payload.get("totalPages")
        reported_size = first_payload.get("size")
    else:
        reported_total = None
        reported_pages = None
        reported_size = None

    first_records = extract_records(first_payload)

    print(f"API-reported total: {reported_total}")
    print(f"API-reported page size: {reported_size}")
    print(f"API-reported pages: {reported_pages}")

    if first_records is None:
        print("No paginated record array identified.")
        print("Response preview:")
        print(json.dumps(first_payload, ensure_ascii=False)[:2000])
        return records, errors

    # Prefer API metadata, but never exceed max_pages.
    if isinstance(reported_pages, int) and reported_pages >= 0:
        page_count = min(reported_pages, max_pages)
    else:
        page_count = max_pages

    start = time.time()

    with tqdm(
        total=page_count,
        desc=name,
        unit="page",
    ) as progress:

        for page in range(1, page_count + 1):
            page_start = time.time()

            try:
                payload = (
                    first_payload
                    if page == 1
                    else get_json(
                        f"{BASE_URL}/{endpoint.lstrip('/')}",
                        params={**base_params, "page": page},
                    )
                )
            except Exception as exc:
                message = f"{name}, page {page}: {exc}"
                errors.append(message)
                tqdm.write(message)
                break

            page_records = extract_records(payload)

            if page_records is None:
                errors.append(
                    f"{name}, page {page}: no record array found."
                )
                tqdm.write(errors[-1])
                break

            if not page_records:
                tqdm.write(f"Page {page} is empty. Stopping.")
                progress.update(1)
                break

            new_count = 0
            duplicates = 0

            for item in page_records:
                if not isinstance(item, dict):
                    continue

                item_id = item.get("id")

                if item_id is not None:
                    item_id = str(item_id)
                    if item_id in seen_ids:
                        duplicates += 1
                        continue
                    seen_ids.add(item_id)

                records.append(item)
                new_count += 1

            progress.update(1)
            progress.set_postfix({
                "total": f"{len(records):,}",
                "new": f"{new_count:,}",
            })

            tqdm.write(
                f"{name} | Page {page}/{page_count} | "
                f"Received {len(page_records):,} | "
                f"New {new_count:,} | "
                f"Duplicates {duplicates:,} | "
                f"Total {len(records):,} | "
                f"{time.time() - page_start:.2f}s"
            )

            if page % CHECKPOINT_EVERY == 0:
                save_records(name, records)
                tqdm.write("Checkpoint saved.")

            # Respect the latest pagination metadata if it changes.
            if isinstance(payload, dict):
                latest_pages = payload.get("totalPages")
                if (
                    isinstance(latest_pages, int)
                    and latest_pages >= 0
                    and page >= latest_pages
                ):
                    break

            time.sleep(0.1)

    print(
        f"Finished {name}: {len(records):,} records "
        f"in {time.time() - start:.1f}s"
    )

    return records, errors


# ---------------------------------------------------------
# 1. ALL LISTINGS
# ---------------------------------------------------------

listings, listing_errors = fetch_paginated(
    name="listings",
    endpoint="listings",
    base_params={
        "limit": REQUESTED_LIMIT,
        "imageVariant": "small",
        "priceGte": 1,
    },
)

# ---------------------------------------------------------
# 2. ALL AREAS
# ---------------------------------------------------------
# Deliberately omit propertyFilter so areas without available
# properties are not excluded by that filter.

areas, area_errors = fetch_paginated(
    name="areas",
    endpoint="areas",
    base_params={
        "limit": REQUESTED_LIMIT,
        "imageVariant": "small",
    },
)

# ---------------------------------------------------------
# 3. PROPERTY CATEGORIES
# ---------------------------------------------------------
# The observed request did not specify a limit.
# Try a large limit; inspect the response to see if supported.

categories, category_errors = fetch_paginated(
    name="property-categories",
    endpoint="property-categories",
    base_params={
        "limit": REQUESTED_LIMIT,
    },
)

# ---------------------------------------------------------
# 4. LISTINGS SUMMARY
# ---------------------------------------------------------
# This endpoint may return an aggregate object rather than
# a paginated list. Preserve its full response as JSON.

print(f"\n{'=' * 70}")
print("FETCHING: LISTINGS SUMMARY")
print("=" * 70)

summary = None
summary_error = None

try:
    summary = get_json(
        f"{BASE_URL}/listings/summary",
        params={
            "page": 1,
            "priceGte": 1,
        },
    )

    summary_path = OUTPUT_DIR / "listings_summary.json"

    with open(summary_path, "w", encoding="utf-8") as file:
        json.dump(
            summary,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print("Summary retrieved successfully.")
    print(f"Saved to: {summary_path}")
    print(json.dumps(summary, ensure_ascii=False, indent=2)[:3000])

except Exception as exc:
    summary_error = str(exc)
    print(f"Summary request failed: {exc}")

# ---------------------------------------------------------
# 5. SAVE FINAL DATASETS
# ---------------------------------------------------------

print(f"\n{'=' * 70}")
print("SAVING FINAL DATASETS")
print("=" * 70)

listings_df = save_records("listings", listings)
areas_df = save_records("areas", areas)
categories_df = save_records("property_categories", categories)

# ---------------------------------------------------------
# 6. COLLECTION REPORT
# ---------------------------------------------------------

all_errors = (
    listing_errors
    + area_errors
    + category_errors
)

if summary_error:
    all_errors.append(f"Summary: {summary_error}")

print(f"\n{'=' * 70}")
print("FINAL COLLECTION REPORT")
print("=" * 70)
print(f"Listings: {len(listings):,}")
print(f"Areas: {len(areas):,}")
print(f"Property categories: {len(categories):,}")
print(f"Errors: {len(all_errors)}")

if all_errors:
    print("\nErrors:")
    for error in all_errors:
        print(f"- {error}")

print("\nOutput directory:", OUTPUT_DIR.resolve())

print("\nListings preview:")
if not listings_df.empty:
    display(listings_df.head())

print("\nAreas preview:")
if not areas_df.empty:
    display(areas_df.head())

print("\nProperty categories preview:")
if not categories_df.empty:
    display(categories_df.head())



FETCHING: LISTINGS
Endpoint: listings
Requested limit: 5000
API-reported total: 992
API-reported page size: 500
API-reported pages: 2


listings:   0%|          | 0/2 [00:00<?, ?page/s]

listings | Page 1/2 | Received 500 | New 500 | Duplicates 0 | Total 500 | 0.00s
listings | Page 2/2 | Received 492 | New 492 | Duplicates 0 | Total 992 | 0.69s
Finished listings: 992 records in 0.8s

FETCHING: AREAS
Endpoint: areas
Requested limit: 5000
API-reported total: 139
API-reported page size: 500
API-reported pages: 1


areas:   0%|          | 0/1 [00:00<?, ?page/s]

areas | Page 1/1 | Received 139 | New 139 | Duplicates 0 | Total 139 | 0.00s
Finished areas: 139 records in 0.0s

FETCHING: PROPERTY-CATEGORIES
Endpoint: property-categories
Requested limit: 5000
API-reported total: 7
API-reported page size: 500
API-reported pages: 1


...

,id,name,slug,streetName,streetNumber,description,longitude,latitude,primaryTag,tags,...,unitTypes,priceRange,area_id,area_name,area_subRegion_name,area_subRegion_region_name,area_subRegion_region_country_name,propertyCategory_name,currency_code,currency_symbol
0,ef65dade-9562-4040-b3fe-7846c88760a9,Spring Court,spring-court-kimbo,None,None,None,36.983417,-1.164222,Student Friendly,"[Affordable, Unfurnished]",...,[{'id': '8cd19e98-5579-4d16-a2d1-f33f1ea6399e'...,"[{'purpose': 'Rent', 'min': 12000, 'max': 12000}]",b0db7b0b-bd8a-47b8-9daa-0f6fb1c599bd,Kimbo,Ruiru,Kiambu,Kenya,Apartment,KES,KSh
1,4f2382ef-8a84-415a-bea2-8cbcb5d67cf5,Tsavo Apartment,tsavo-apartment-fedha,None,None,None,36.899938,-1.318739,Family Friendly,"[Mid-Range, Unfurnished]",...,[{'id': '443734cb-f132-45ac-89db-a1daea87075c'...,"[{'purpose': 'Rent', 'min': 18000, 'max': 18000}]",e5945e87-d382-4449-a80c-44f62783df97,Fedha,Embakasi,Nairobi,Kenya,Apartment,KES,KSh
2,1bdd1ff1-dc62-4264-906c-a477129ae6f1,Haven Heights,haven-heights-fedha,None,None,None,36.905011,-1.316651,Family Friendly,"[Mid-Range, Unfurnished]",...,[{'id': 'ec21b2c5-bbef-456a-8d8a-50f9865b34a5'...,"[{'purpose': 'Rent', 'min': 45000, 'max': 55000}]",e5945e87-d382-4449-a80c-44f62783df97,Fedha,Embakasi,Nairobi,Kenya,Apartment,KES,KSh
3,7e99b029-0c94-4fce-97d3-370423a969bf,Karama Height,karama-height-fedha,None,None,None,36.901817,-1.316767,Family Friendly,"[Mid-Range, Unfurnished]",...,[{'id': 'ec21b2c5-bbef-456a-8d8a-50f9865b34a5'...,"[{'purpose': 'Rent', 'min': 30000, 'max': 30000}]",e5945e87-d382-4449-a80c-44f62783df97,Fedha,Embakasi,Nairobi,Kenya,Apartment,KES,KSh
4,af1d557f-adbc-4c8a-bc55-e8058be36470,Apex Apartment,apex-apartment-fedha,None,None,None,36.897063,-1.308381,Family Friendly,"[Affordable, Unfurnished]",...,[{'id': 'ec21b2c5-bbef-456a-8d8a-50f9865b34a5'...,"[{'purpose': 'Rent', 'min': 23000, 'max': 23000}]",e5945e87-d382-4449-a80c-44f62783df97,Fedha,Embakasi,Nairobi,Kenya,Apartment,KES,KSh



Areas preview:


,id,name,slug,imageUrl,imageBlurHash,subRegionId,description,createdAt,updatedAt,deletedAt,numberOfListing,subRegion_id,subRegion_name,region_id,region_name,country_id,country_name
0,3bf8ba46-c319-4d9f-b517-5dc7a50a8b2c,Mlolongo,mlolongo,https://assets.reemioltd.com/areas_images/0e29...,"UdDd0J%1jZt7cIxtoeayx^xuodR,R-flt5Wr",a71b3125-5cb8-4d2d-a7a8-f0231ea8b275,,2026-08-12T10:01:55.444Z,2026-10-07T08:47:42.670Z,None,2,a71b3125-5cb8-4d2d-a7a8-f0231ea8b275,Athi River,9825d201-707a-49bc-9e9d-ebfc3108262f,Machakos,635c2f7f-94b2-4176-8c9a-92a9d4925f1a,Kenya
1,95c28311-7114-4620-a451-a217a86ee5e0,Athi River,athi-river,https://assets.reemioltd.com/areas_images/d0e9...,UaFiS:%MW?t7yGt8t7j]E9ayxajtD+obobWX,a71b3125-5cb8-4d2d-a7a8-f0231ea8b275,,2026-07-27T11:55:42.462Z,2026-07-27T11:55:45.810Z,None,4,a71b3125-5cb8-4d2d-a7a8-f0231ea8b275,Athi River,9825d201-707a-49bc-9e9d-ebfc3108262f,Machakos,635c2f7f-94b2-4176-8c9a-92a9d4925f1a,Kenya
2,137d6c24-ef1e-4574-9d74-9f3c8e6da35f,Kirigiti,kirigiti,https://assets.reemioltd.com/areas_images/7319...,UgGb}B?Z%Max%%xos+t8x^xus7t8%Nxvn}WB,52ecd24c-3d83-423f-955e-742c72fefe9d,,2026-05-22T07:34:54.199Z,2026-05-22T07:34:56.697Z,None,15,52ecd24c-3d83-423f-955e-742c72fefe9d,Kiambu,1ccf6175-ed3a-4f8f-ab99-c6b2ee5c37e6,Kiambu,635c2f7f-94b2-4176-8c9a-92a9d4925f1a,Kenya
3,ab3034f5-59c9-4abe-9eae-e281671eb178,Lenana,lenana,https://assets.reemioltd.com/areas_images/12a5...,"UeD,ZsxUohRj.AxZbHWCbJofaxtSa~WAayt8",0d834b9f-121d-4278-9554-fc73f41fc472,,2026-05-20T10:56:53.925Z,2026-05-20T10:56:56.577Z,None,1,0d834b9f-121d-4278-9554-fc73f41fc472,Dagoretti,ed3f6986-bfdf-4aa2-8d01-b7c4517450ef,Nairobi,635c2f7f-94b2-4176-8c9a-92a9d4925f1a,Kenya
4,40c8eef5-e635-4c5d-b621-be18c99ae6be,Kiambu,kiambu,https://assets.reemioltd.com/areas_images/aba6...,UxF?O+xvt7t7?wozofa#bdRjayWBofRia#of,52ecd24c-3d83-423f-955e-742c72fefe9d,,2026-05-18T08:49:31.437Z,2026-05-18T08:49:33.953Z,None,25,52ecd24c-3d83-423f-955e-742c72fefe9d,Kiambu,1ccf6175-ed3a-4f8f-ab99-c6b2ee5c37e6,Kiambu,635c2f7f-94b2-4176-8c9a-92a9d4925f1a,Kenya



Property categories preview:


,id,name,slug,imageUrl,imageBlurHash,description,createdAt,deletedAt,updatedAt
0,a693b07f-ccb9-41f0-8393-f4cfcb34619b,Mixed Development,mixed-development,https://assets.reemioltd.com/property_categori...,URI=42Md.8%M.TM{-;oz_3?bM{t89vtRaeay,,2025-12-19T12:12:37.497Z,None,2026-09-21T12:33:12.864Z
1,56d2d058-9b34-4022-b7fc-337eea61a536,Houses,houses,https://assets.reemioltd.com/property_categori...,"UfJ7,ZWBIpx]Y8Rjj[tR_3WAaet7JCRjjFWC",,2025-12-16T07:29:28.396Z,None,2026-09-30T08:22:07.190Z
2,a84b2f50-a380-4b8a-9055-22e5e609274a,Loft,loft,None,None,None,2025-12-16T06:59:18.113Z,None,2025-12-16T06:59:18.113Z
3,36af1301-5292-4691-864e-ac58a4e70e1a,Townhouse,townhouse,https://assets.reemioltd.com/property_categori...,UBG8yK000VtS5Jit.At95^=^bJI_S+Rl-px_,"A multi-floor, single-family home that shares ...",2025-12-15T18:06:15.584Z,None,2026-09-11T10:42:37.292Z
4,1297c7c0-08b4-45fe-a550-fc61104030c9,Bungalow,bungalow,https://assets.reemioltd.com/property_categori...,UQDm5VIUayWB?wRjofWB.8kCayfQWXt7Rjt7,,2025-12-15T18:04:49.768Z,None,2026-09-30T08:25:17.341Z
